# Actor-Critic Methods

Actor-Critic algorithms combine the policy parametrisation of policy gradient
methods with the online TD learning of value-based methods.

| Algorithm | Critic update | Actor update | Bias / Variance |
|-----------|--------------|--------------|------------------|
| **One-step AC** | TD(0) delta per step | delta * grad log pi | Higher bias, lower variance than REINFORCE |
| **AC(lambda)** | TD(lambda) via traces | delta * eligibility trace | Interpolates between TD(0) and Monte-Carlo |

Both use a tabular softmax actor (`theta`, shape 64x9) and a tabular
state-value critic (`V`, shape 64).

> **Note on `abs_err` metric:** same caveat as REINFORCE — `V_est = V^pi`,
> not V*. Use `policy_err` for cross-family comparisons.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from experiments.run_all import (
    load_config, run_algorithm, run_episode,
    _make_actor_critic, _make_actor_critic_lambda,
)
from environment.gridworld import GridWorldEnv
from utils.plotting import (
    plot_multi_curves, plot_learning_curves,
    plot_value_heatmap, plot_vstar_heatmap,
    plot_policy_arrows, plot_summary_bar, save_figure,
)
from utils.metrics import V_STAR, policy_eval_error

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})

## 0. Hyperparameters

In [ ]:
cfg = load_config()

print(f"gamma          = {cfg['environment']['gamma']}")
print(f"n_episodes     = {cfg['environment']['n_episodes']}")
print()
print('Actor-Critic')
print(f"  lr_actor     = {cfg['actor_critic']['lr_actor']}")
print(f"  lambda_critic= {cfg['actor_critic']['lambda_critic']}  (AC(lambda) only)")
print(f"  lambda_actor = {cfg['actor_critic']['lambda_actor']}  (AC(lambda) only)")

## 1. Algorithm Details

### One-step AC — per-step update

```
delta   = cost + gamma * V[s'] - V[s]     # TD error (critic signal)
V[s]   += lr_critic * delta
theta[s, a_t] += lr_actor * delta * (pi[a_t] - 1)   # cost-min convention
theta[s, a]   += lr_actor * delta * pi[a]  for a != a_t
```

### AC(lambda) — eligibility trace update

Traces `z_v` (64,) and `z_theta` (64, 9) are reset to zero at episode start.

```
z_v    *= gamma * lambda_critic          # decay all states
z_v[s] += 1.0

z_theta          *= gamma * lambda_actor
z_theta[s, :]    += pi[s, :]            # cost-min convention: += pi, then -= I_a
z_theta[s, a_t]  -= 1.0

delta   = cost + gamma * V[s'] - V[s]
V      += lr_critic * delta * z_v       # global update via traces
theta  += lr_actor  * delta * z_theta
```

Setting lambda=0 recovers one-step AC. Setting lambda=1 recovers Monte-Carlo.

## 2. Running the Experiments

In [ ]:
nb_cfg = {**cfg, 'environment': {**cfg['environment'], 'n_runs': 3}}

In [ ]:
print('Running one-step Actor-Critic ...')
ac_results = run_algorithm('actor_critic', _make_actor_critic, nb_cfg, verbose=True)

In [ ]:
print('Running AC(lambda) ...')
acl_results = run_algorithm('actor_critic_lambda', _make_actor_critic_lambda, nb_cfg, verbose=True)

## 3. Learning Curves

In [ ]:
results_map = {
    'Actor-Critic': ac_results,
    'AC(lambda)':   acl_results,
}

fig = plot_multi_curves(
    results_map, metric='abs_err',
    title='Absolute Relative Error  (V_est = V^pi)',
    smooth=10,
)
save_figure(fig, '04_abs_err')
plt.show()

In [ ]:
fig = plot_multi_curves(
    results_map, metric='policy_err',
    title='Policy Evaluation Error  (primary metric)',
)
save_figure(fig, '04_policy_err')
plt.show()

## 4. Lambda Sensitivity (AC(lambda))

How sensitive is AC(lambda) to the trace decay rate?
Lambda=0 is one-step TD; lambda=1 is full Monte-Carlo.

In [ ]:
from algorithms.actor_critic import ActorCriticLambdaAgent

lambda_values = [0.0, 0.5, 0.9, 0.99]
lambda_results = {}

for lam in lambda_values:
    def factory_lam(rng, cfg, lam=lam):
        return ActorCriticLambdaAgent(
            rng=rng,
            gamma=cfg['environment']['gamma'],
            lr_critic=cfg['actor_critic']['lr_actor'],
            lr_actor=cfg['actor_critic']['lr_actor'],
            lambda_critic=lam,
            lambda_actor=lam,
        )
    res = run_algorithm(f'ac_lambda_{lam}', factory_lam, nb_cfg, verbose=False)
    lambda_results[f'lambda={lam}'] = res
    print(f'lambda={lam}  final_policy_err={res["policy_err"][:, -1].mean():.4f}')

In [ ]:
fig = plot_multi_curves(
    lambda_results, metric='policy_err',
    title='AC(lambda) — policy_err for different lambda values',
)
save_figure(fig, '04_lambda_sweep')
plt.show()

## 5. Value Function and Policy

In [ ]:
n_eps  = cfg['environment']['n_episodes']
max_st = cfg['environment']['max_steps_per_episode']

agents = {}
for name, factory in [('Actor-Critic', _make_actor_critic),
                       ('AC(lambda)',   _make_actor_critic_lambda)]:
    agent = factory(np.random.default_rng(42), cfg)
    env   = GridWorldEnv(np.random.default_rng(43))
    for ep in range(n_eps):
        run_episode(agent, env, max_st, ep)
    agents[name] = agent
    print(f'{name:<15}  policy_err={policy_eval_error(agent.get_policy()):.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
plot_vstar_heatmap(ax=axes[0])
plot_value_heatmap(agents['Actor-Critic'].get_value_estimate(),
                   title='AC  V^pi(s)', ax=axes[1])
plot_value_heatmap(agents['AC(lambda)'].get_value_estimate(),
                   title='AC(lambda)  V^pi(s)', ax=axes[2])
fig.suptitle('Critic Value Functions after 1500 episodes (V_est = V^pi)',
             fontsize=12, y=1.02)
fig.tight_layout()
save_figure(fig, '04_value_heatmaps')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
plot_policy_arrows(agents['Actor-Critic'].get_policy(),
                   title='Actor-Critic — policy (argmax)', ax=axes[0])
plot_policy_arrows(agents['AC(lambda)'].get_policy(),
                   title='AC(lambda) — policy (argmax)', ax=axes[1])
fig.suptitle('Learned Policies after 1500 episodes', fontsize=13)
fig.tight_layout()
save_figure(fig, '04_policy_arrows')
plt.show()

## 6. Summary

In [ ]:
def final_stats(res, name):
    pe_m = res['policy_err'][:, -1].mean()
    pe_s = res['policy_err'][:, -1].std()
    print(f'{name:<15}  policy_err={pe_m:.4f}+/-{pe_s:.4f}')
    return pe_m

final_pol = {
    'Actor-Critic': final_stats(ac_results,  'Actor-Critic'),
    'AC(lambda)':   final_stats(acl_results, 'AC(lambda)'),
}

fig = plot_summary_bar(final_pol, title='Final Policy Evaluation Error')
save_figure(fig, '04_final_bar')
plt.show()

## Discussion

**One-step Actor-Critic** updates both V and theta after every step using the
one-step TD error delta. Compared to REINFORCE, it has lower variance
(single-step signal instead of full-episode return) but higher bias (the TD
target `cost + gamma * V[s']` is only a one-step approximation to G_t).

**AC(lambda)** uses eligibility traces to accumulate credit for past
state-action pairs. When a large TD error occurs, it propagates backward
through the trace, updating states visited recently proportional to how
recently they were visited. Lambda controls the length of the credit:

- **lambda = 0**: identical to one-step AC (traces reset immediately)
- **lambda = 0.9** (default): medium-length credit; good empirical trade-off
- **lambda = 1.0**: equivalent to Monte-Carlo, but with online updates

**Implementation note:** the eligibility traces must decay *all* states at
each step (`z_v *= gamma * lambda`), not just the current state. Updating
only the current state leaves stale large values in other states that then
participate in the global `V += lr * delta * z_v` update, causing V to
diverge.